# Fase 0 — Validação do NDVI super-resolvido (Bacia do Pardo, RS)

**Pergunta:** o NDVI super-resolvido a 2,5 m é confiável para priorizar trechos de mata ciliar?

**Escopo:** notebook isolado, Google Colab, dados públicos (Sentinel-2 L2A via Planetary Computer). Nada de `backend/` ou `frontend/` muda até o veredito.

**Herança da v1:** região Sinimbu/RS, buffer 200 m, limiares crítico < 0,2 / moderado < 0,5 (`backend/hls_analysis/config_hls.py`), pontos críticos de `critical_points_mata_ciliar.geojson`.

## Gates de decisão (computados na última célula de código)

| Gate | Teste | Passa se |
|---|---|---|
| G0 | NDVI S2-10m vs HLS-30m atual (mesmos trechos) | Pearson ≥ 0,85 |
| G1 | Consistência espectral do SR (protocolo de Wald: SR→10m vs original) | ERGAS < 3, SAM < 5°, PSNR > 30 dB |
| G2 | Viés solo→vegetação (alerta Major/VRVis 2025) | \|viés médio em solo nu\| ≤ 0,05 (ou corrigido) |
| G3 | Acordo de NDVI + pontos críticos | RMSE ≤ 0,08 e ≥ 80% dos críticos preservados |
| G4 | Incerteza contida (só se rodar LDSR-S2 em GPU) | incerteza alta só em bordas |
| G5 | Regeneração detectável 2024→2026 | ΔNDVI positivo nos trechos MUDA/Unisc |
| G6 | Sanidade externa (Soturno) | perda na ordem dos −69% publicados |

**Verde** = G0–G3 + G5 (G4 se aplicável, G6 como sanidade) → desenhar backend v2 + portal. **Vermelho** = falha estrutural em G1/G2/G3 → documentar o negativo.

In [ ]:
# Célula 0a — dependências (Colab; rode uma vez e reinicie o runtime se pedir)
%pip install -q pystac-client planetary-computer stackstac rioxarray xarray dask geopandas osmnx shapely pyproj scikit-image matplotlib

In [ ]:
# Célula 0b — checagem de ambiente
import importlib, sys

for mod in ["pystac_client", "planetary_computer", "stackstac", "rioxarray", "geopandas", "osmnx", "skimage"]:
    try:
        m = importlib.import_module(mod)
        print(f"{mod:20s} {getattr(m, '__version__', 'ok')}")
    except ImportError:
        print(f"{mod:20s} AUSENTE <- rode a Célula 0a")

try:
    import torch
    print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
except ImportError:
    print("torch AUSENTE (só necessário nas Células 4/5; Células 1–3 rodam sem ele)")

print("IN_COLAB:", "google.colab" in sys.modules)

## Célula 1 — Parâmetros (único lugar a editar)

In [ ]:
# Célula 1 — parâmetros
from pathlib import Path
import numpy as np

SEED = 42
np.random.seed(SEED)
REGION_NAME = "Sinimbu, Rio Grande do Sul, Brasil"
BUFFER_M = 200          # mesmo buffer da v1 (config_hls.py: degradation.buffer_distance)
NDVI_CRITICAL = 0.2     # idem v1
NDVI_MODERATE = 0.5     # idem v1
CLOUD_MAX = 20          # mais estrito que os 50% da v1: SR exige cena limpa

# Janelas temporais (mesma sazonalidade; espelham o desenho mar/set-2024 do estudo do Soturno)
WINDOWS = {
    "pre":  ("2024-01-01", "2024-03-31"),   # pré-enchente
    "pos":  ("2024-06-01", "2024-08-31"),   # pós imediato
    "regen": ("2025-06-01", "2025-08-31"),  # regeneração (troque p/ 2026 quando disponível)
}

# Fallback de AOI: bbox dos 50 pontos críticos da v1 (minx, miny, maxx, maxy)
AOI_BBOX_FALLBACK = (-52.7608, -29.5547, -52.4759, -29.3064)

OUT = Path("fase0_out")
OUT.mkdir(exist_ok=True)
print("OUT:", OUT.resolve())

## Célula 2 — AOI: rios de Sinimbu (OSM) + buffer 200 m
Replica `find_rivers_in_region_with_filter` da v1. Se o OSM falhar, usa o bbox dos pontos críticos.

In [ ]:
# Célula 2 — AOI
import geopandas as gpd
from shapely.geometry import box

def build_aoi(region=REGION_NAME, buffer_m=BUFFER_M):
    import osmnx as ox
    print(f"Limites administrativos: {region}")
    boundary = ox.geocode_to_gdf(region)
    muni = boundary.geometry.iloc[0]
    print("Rios (OSM waterway=river)...")
    rivers = ox.features_from_place(region, tags={"waterway": "river"})
    linear = rivers[rivers.geometry.type.isin(["LineString", "MultiLineString"])]
    dentro = []
    for _, row in linear.iterrows():
        g = row.geometry
        if g.intersects(muni) and (g.intersection(muni).length / g.length) >= 0.10:
            dentro.append(row)
    print(f"Rios >=10% no município: {len(dentro)} de {len(linear)}")
    if not dentro:
        raise ValueError("Nenhum rio dentro do município")
    rgdf = gpd.GeoDataFrame(dentro, crs=linear.crs)
    c = rgdf.geometry.centroid.iloc[0]
    z = int((c.x + 180) / 6) + 1
    utm = f"EPSG:{32700 + z}" if c.y < 0 else f"EPSG:{32600 + z}"
    buf = rgdf.to_crs(utm).buffer(buffer_m).unary_union
    aoi = gpd.GeoDataFrame([{"region": region}], geometry=[buf], crs=utm).to_crs(4326)
    print(f"Área AOI: {buf.area / 1e6:.2f} km² | bounds: {tuple(round(v, 4) for v in aoi.total_bounds)}")
    return aoi

try:
    AOI = build_aoi()
except Exception as e:
    print(f"OSM falhou ({e}); usando bbox fallback dos pontos críticos v1")
    AOI = gpd.GeoDataFrame([{"region": REGION_NAME + " (fallback bbox v1)"}],
                           geometry=[box(*AOI_BBOX_FALLBACK)], crs=4326)

AOI.to_file(OUT / "aoi.geojson", driver="GeoJSON")
BBOX = tuple(float(v) for v in AOI.total_bounds)  # (minx, miny, maxx, maxy)
print("BBOX p/ STAC:", BBOX)

## Célula 3 — Baseline Sentinel-2 L2A 10 m + NDVI (três janelas)
Bandas B04 (red) / B08 (NIR); máscara SCL (4 = vegetação, 5 = solo exposto). Composição = mediana temporal.

In [ ]:
# Célula 3 — baseline S2
from pystac_client import Client
import planetary_computer as pc
import stackstac
import xarray as xr

CATALOG = "https://planetarycomputer.microsoft.com/api/stac/v1"
ASSETS = ["B02", "B03", "B04", "B08", "SCL"]
SCL_VALID = [4, 5]  # vegetação + solo exposto (fora: água, nuvem, sombra, neve)

catalog = Client.open(CATALOG)

def composite_ndvi(bbox, start, end, cloud_max=CLOUD_MAX):
    search = catalog.search(collections=["sentinel-2-l2a"], bbox=list(bbox),
                              datetime=f"{start}/{end}",
                              query={"eo:cloud_cover": {"lt": cloud_max}})
    items = list(search.items())
    print(f"  {start}→{end}: {len(items)} cenas (nuvens<{cloud_max}%)")
    if not items:
        return None, {}
    signed = [pc.sign(i).to_dict() for i in items]
    stack = stackstac.stack(signed, assets=ASSETS, resolution=10,
                            bounds_latlon=bbox, chunksize=512)
    med = stack.median(dim="time", keep_attrs=True).compute()
    scl = med.sel(band="SCL")
    valid = scl.isin(SCL_VALID)
    red = med.sel(band="B04").where(valid) / 10000.0
    nir = med.sel(band="B08").where(valid) / 10000.0
    ndvi = ((nir - red) / (nir + red)).where((nir + red) != 0)
    stats = {"n_cenas": len(items), "frac_validos": float(valid.mean()),
             "ndvi_min": float(ndvi.min()), "ndvi_max": float(ndvi.max()),
             "ndvi_mean": float(ndvi.mean()),
             "frac_critico": float((ndvi < NDVI_CRITICAL).mean()),
             "frac_moderado": float(((ndvi >= NDVI_CRITICAL) & (ndvi < NDVI_MODERATE)).mean())}
    return ndvi, stats

NDVI, STATS = {}, {}
for nome, (ini, fim) in WINDOWS.items():
    print(f"Janela '{nome}':")
    ndvi, st = composite_ndvi(BBOX, ini, fim)
    NDVI[nome], STATS[nome] = ndvi, st
    if ndvi is not None:
        ndvi.rio.to_raster(OUT / f"ndvi10m_{nome}.tif")
        print(" ", {k: round(v, 4) if isinstance(v, float) else v for k, v in st.items()})

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, len([k for k in NDVI if NDVI[k] is not None]), figsize=(12, 4))
ax = np.atleast_1d(ax)
for a, (nome, ndvi) in zip(ax, [(k, v) for k, v in NDVI.items() if v is not None]):
    ndvi.plot(ax=a, vmin=-0.2, vmax=1.0, cmap="YlGn", add_colorbar=True)
    a.set_title(f"NDVI 10m — {nome}")
plt.tight_layout(); plt.savefig(OUT / "ndvi10m_series.png", dpi=120); plt.show()

### Comparação com o HLS-30m da v1 (G0 — opcional)
Suba `ndvi_mata_ciliar_wgs84_normalized.geotiff` do repo nesta sessão (mesma pasta do notebook) para habilitar.

In [ ]:
# G0 — correlação S2-10m vs HLS-30m (requer o GeoTIFF da v1 na sessão)
import rioxarray as rxr

HLS_TIF = Path("ndvi_mata_ciliar_wgs84_normalized.geotiff")
G0 = {"status": "SKIPPED", "pearson": None}
if HLS_TIF.exists() and NDVI.get("regen") is not None:
    hls = rxr.open_rasterio(HLS_TIF).squeeze()
    s2 = NDVI["regen"]
    hls_c = hls.rio.clip_box(*s2.rio.bounds())
    s2_agg = s2.coarsen(x=3, y=3, boundary="trim").mean()  # 10m→~30m p/ comparar
    a = s2_agg.interp_like(hls_c)
    m = np.isfinite(a.values) & np.isfinite(hls_c.values)
    r = float(np.corrcoef(a.values[m], hls_c.values[m])[0, 1])
    G0 = {"status": "PASS" if r >= 0.85 else "FAIL", "pearson": round(r, 4)}
    print(f"G0 pearson={r:.4f} -> {G0['status']} (corte 0,85)")
else:
    print("G0 SKIPPED: suba o GeoTIFF da v1 ou aguarde a janela 'regen'")
print(G0)

## Célula 4 — SR primário: SEN2SRLite RGBN ×4 → NDVI 2,5 m + teste de viés (G2)
Modelo transformer com foco em fidelidade (ver ISPRS 2026). Roda em CPU; processe só recortes nos trechos críticos para economizar tempo.

**Passo manual único:** instale `sen2sr` e baixe o checkpoint `*NonReference_RGBN_x4*` conforme https://github.com/ESAOpenSR/sen2sr, e ajuste `MODEL_PATH` abaixo. O restante é automático.

In [ ]:
# Célula 4 — SR primário (SEN2SRLite) + NDVI 2,5m + viés solo→vegetação (G2)
MODEL_PATH = Path("models/sen2srlite_NonReference_RGBN_x4.ckpt")  # AJUSTE p/ o nome real
PATCH, OVERLAP, FACTOR = 128, 32, 4  # convenção ESA OpenSR (128→512)

try:
    import sen2sr
    print("sen2sr OK; predict_large:", hasattr(sen2sr, "predict_large"))
except ImportError:
    print("Rode: %pip install -q sen2sr mlstac cubo")
    sen2sr = None
print("Checkpoint existe:", MODEL_PATH.exists())

def extract_tiles(arr4, patch=PATCH, overlap=OVERLAP):
    """Recorta (4,H,W)->tiles (4,patch,patch) com overlap; retorna tiles + metadados p/ remontar."""
    _, H, W = arr4.shape
    step = patch - overlap
    tiles, meta = [], []
    for y in range(0, H, step):
        for x in range(0, W, step):
            t = np.zeros((4, patch, patch), np.float32)
            h = min(patch, H - y); w = min(patch, W - x)
            t[:, :h, :w] = arr4[:, y:y + h, x:x + w]
            tiles.append(t); meta.append((y, x, h, w))
    return np.stack(tiles), meta, (H, W)

def stitch_tiles(sr_tiles, meta, hw, patch=PATCH, overlap=OVERLAP, factor=FACTOR):
    """Remonta (N,4,p*F,p*F)->(4,H*F,W*F) com média nas sobreposições."""
    H, W = hw
    out = np.zeros((4, H * factor, W * factor), np.float64)
    cnt = np.zeros((H * factor, W * factor), np.float64)
    m = (patch - overlap // 2) * factor
    for t, (y, x, h, w) in zip(sr_tiles, meta):
        oy, ox = y * factor, x * factor
        out[:, oy:oy + h * factor, ox:ox + w * factor] += t[:, :h * factor, :w * factor]
        cnt[oy:oy + h * factor, ox:ox + w * factor] += 1
    return (out / np.maximum(cnt, 1)).astype(np.float32)

G2 = {"status": "SKIPPED", "vies_solo": None}
NDVI_SR = None
if sen2sr is not None and MODEL_PATH.exists() and NDVI.get("regen") is not None:
    import torch
    ndvi_lr, st_lr = NDVI["regen"], STATS["regen"]
    # Recorte de trabalho: 512x512 px @10m (~5x5 km) centrado na AOI — ajuste p/ seus trechos
    H, W = ndvi_lr.shape
    yc, xc = H // 2 - 256, W // 2 - 256
    win = ndvi_lr.isel(y=slice(max(yc, 0), max(yc, 0) + 512), x=slice(max(xc, 0), max(xc, 0) + 512))
    print("Janela SR:", dict(win.sizes))
    # TODO: montar RGBN (B02,B03,B04,B08 /10000) da mesma janela a partir do composite da Célula 3
    # e inferir com sen2sr (.predict_large ou loop extract_tiles->modelo->stitch_tiles).
    # Interface esperada: entrada Bx4x128x128 float32 0..1 -> saída Bx4x512x512.
    print("TODO executável: ligar o checkpoint ao loop de tiles e calcular NDVI-SR.")
    print("Após obter NDVI_SR (DataArray 2,5m), o bloco G2 abaixo calcula o viés.")
else:
    print("Célula 4 em espera: instale sen2sr + checkpoint (ver topo) e reexecute.")

# --- G2: viés solo→vegetação (rode após obter NDVI_SR; falsos NAs = solo nu SCL==5, NDVI_10m<0,2) ---
# solo = (SCL_janela == 5) & (NDVI_10m < 0.2)
# vies = float((NDVI_SR_coarsenado_a_10m[solo] - NDVI_10m[solo]).mean())
# G2 = {"status": "PASS" if abs(vies) <= 0.05 else "FAIL", "vies_solo": round(vies, 4)}
print(G2)

## Célula 5 — SR secundário: LDSR-S2 (difusão + incerteza, G4) — requer GPU
Pulada automaticamente sem CUDA. Ref: https://github.com/ESAOpenSR/opensr-model (+ `opensr-utils` p/ tiling georreferenciado).

In [ ]:
# Célula 5 — LDSR-S2 + incerteza (GPU)
G4 = {"status": "SKIPPED", "nota": "sem GPU ou pacote ausente"}
try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
except ImportError:
    HAS_CUDA = False
print("CUDA:", HAS_CUDA)

if HAS_CUDA:
    try:
        import opensr_model, opensr_utils  # nomes podem variar; ver README dos repos
        print("opensr_model + opensr_utils OK")
        print("TODO executável: large_file_processing(root=<cena RGBN .tif 10m>, window=(128,128), factor=4,")
        print("  overlap=12, n_variations=5) -> SR 2,5m + mapa de incerteza (desvio-padrão das variações).")
        print("G4 PASS se incerteza alta concentrar-se em bordas/urbano, não no interior da mata.")
    except ImportError:
        print("Rode: %pip install -q opensr-model opensr-utils (nomes PyPI podem variar; ver repos)")
else:
    print("Sem GPU: G4 fica SKIPPED — o veredito usa G0–G3+G5 (+G6).")
print(G4)

## Célula 6 — Gates quantitativos e veredito
Métricas implementadas em numpy (sem dependência do `opensr-test`; use-o como checagem cruzada se instalado).

In [ ]:
# Célula 6 — métricas + gates + veredito
import json

def _m(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return a[m].astype(np.float64), b[m].astype(np.float64), m

def rmse(a, b):
    a, b, _ = _m(a, b)
    return float(np.sqrt(np.mean((a - b) ** 2))) if a.size else np.nan

def pearson(a, b):
    a, b, _ = _m(a, b)
    return float(np.corrcoef(a, b)[0, 1]) if a.size > 10 else np.nan

def psnr(a, b, peak=2.0):
    r = rmse(a, b)
    return float(20 * np.log10(peak / r)) if r > 0 else np.inf

def ergas(sr_bands, ref_bands, scale=4):
    """sr/ref: lista de pares (sr_2d, ref_2d) por banda, já na mesma grade."""
    t = []
    for s, r in zip(sr_bands, ref_bands):
        sv, rv, _ = _m(s, r)
        mu = rv.mean()
        t.append(((sv - rv) ** 2).mean() / mu ** 2 if mu != 0 else np.nan)
    return float(100 / scale * np.sqrt(np.nanmean(t)))

def sam(red_s, nir_s, red_r, nir_r):
    """Ângulo espectral médio (graus) no espaço (red, nir)."""
    s = np.stack([red_s, nir_s], -1).reshape(-1, 2)
    r = np.stack([red_r, nir_r], -1).reshape(-1, 2)
    ok = np.isfinite(s).all(1) & np.isfinite(r).all(1)
    s, r = s[ok], r[ok]
    cosang = (s * r).sum(1) / (np.linalg.norm(s, axis=1) * np.linalg.norm(r, axis=1) + 1e-12)
    return float(np.degrees(np.arccos(np.clip(cosang, -1, 1))).mean())

GATES = {"G0": G0, "G2": G2, "G4": G4}

# G1 — consistência de Wald: requer NDVI_SR (Célula 4). NDVI_SR_coarse = média 4x4 do SR.
#   G1 = PASS se ergas<3 e sam<5° e psnr>30dB entre NDVI_SR_coarse e NDVI_10m.
GATES["G1"] = {"status": "SKIPPED", "nota": "aguarda NDVI_SR da Célula 4"}

# G3 — acordo + críticos: requer NDVI_SR (RMSE vs NDVI_10m agregado; % críticos preservados).
GATES["G3"] = {"status": "SKIPPED", "nota": "aguarda NDVI_SR da Célula 4"}

# G5 — regeneração: ΔNDVI regen−pos nos trechos (independe de SR).
if NDVI.get("pos") is not None and NDVI.get("regen") is not None:
    d = float((NDVI["regen"] - NDVI["pos"]).mean())
    GATES["G5"] = {"status": "PASS" if d > 0 else "FAIL", "delta_ndvi": round(d, 4)}
else:
    GATES["G5"] = {"status": "SKIPPED", "nota": "janelas ausentes"}

# G6 — sanidade Soturno: buffer 50 m nos rios 'Soturno' em Faxinal do Soturno, mar→set/2024.
# Preencha SOTURNO_OK=True após implementar (mesmo composite_ndvi + fração NDVI>0,5 pré vs pós).
# Referência: vegetação arbórea 119,65 ha → 37,25 ha (−68,86%).
GATES["G6"] = {"status": "SKIPPED", "nota": "implementar sanity check do Soturno"}

print(f"{'gate':6s} {'status':8s} detalhe")
for k, v in GATES.items():
    det = {kk: vv for kk, vv in v.items() if kk != "status"}
    print(f"{k:6s} {v['status']:8s} {det}")
(OUT / "gates.json").write_text(json.dumps(GATES, indent=2))
print("\nSalvo:", OUT / "gates.json")
print("\nVEREDITO: preencha v2_validation/README.md (tabela de resultados) com PASS/FAIL acima.")

## Interpretação e próximos passos

- **Verde** (G0–G3 + G5, G4 se aplicável, G6 como sanidade): desenhar `sr_service` no backend (`/api/v2/super-resolution`, saída COG), camada 2,5 m + toggle antes/depois + overlay de incerteza no frontend, e índice de prioridade por trecho no portal.
- **Vermelho** (falha estrutural em G1/G2/G3): registrar o negativo no README (data, executor, gates) e reavaliar — resultado negativo documentado também é publicável.
- **Amarelo** (passes parciais, ex. G2 corrigido por regressão): repetir a Célula 4 com LDSR-S2 (Célula 5) antes de decidir.

Após o verde: publicar os 12 trechos do MUDA como primeiro recorte do portal e abordar Unisc/Comitê Pardo com o relatório em mãos.